# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzlanFaisalRaj/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Signal check 1 — staleness (behind the refresh flags)

Bucket: `freshness_tier` (built from `days_since_last_update`) vs `is_declining_label`
(`trend_direction == "down"`). Overall base rate across all 30,000 rows: **54.2%**.

| freshness_tier | n | decline_rate |
|---|---|---|
| 0-30 | 20,480 | 51.1% |
| 31-90 | 175 | 58.9% |
| 91-180 | 9,171 | **61.1%** |
| 181+ | 174 | 47.1% |

**Verdict: MIXED.** On the two buckets with real sample size, staleness does push decline up —
`91-180` (n=9,171) sits at 61.1%, clearly above both the `0-30` fresh bucket (51.1%, n=20,480)
and the 54.2% base rate. But the signal does **not** hold at the extreme tail: `181+` (n=174)
drops back to 47.1%, below baseline. That bucket is small enough (174 rows) that I don't trust
it in either direction, but it means I can't claim "the older, the worse" as a straight line —
only that the `91-180` window specifically carries elevated risk. My rule uses that window, not
"any staleness."

### Signal check 2 — CTR vs position (behind the CTR-fix logic)

First pass, no volume floor — misleading:

| position_tier | n | mean_ctr | median_ctr |
|---|---|---|---|
| top_3 | 2,321 | 1.48 | 0.00 |
| page_1 | 11,814 | 0.65 | 0.16 |
| striking | 7,304 | 0.32 | 0.11 |
| page_3_5 | 7,242 | 0.22 | 0.03 |
| deep | 1,319 | 0.15 | 0.00 |

`top_3` median CTR of 0.00 looked backwards until I checked volume: `top_3`'s median
`impressions_90d` is only **3**, and 76.8% of `top_3` rows have zero clicks — these are pages
ranked #1–3 for near-zero-search queries, not real winners. The data dictionary warns exactly
about this ("never read a tier median without stating its volume floor"), so I refiltered to
`impressions_90d >= 300` (the "moderate" visibility floor already defined in the dictionary):

| position_tier | n | median_ctr (floor-filtered) |
|---|---|---|
| page_1 | 7,623 | 0.23 |
| top_3 | 485 | 0.20 |
| striking | 5,078 | 0.17 |
| page_3_5 | 5,004 | 0.08 |
| deep | 562 | 0.00 |

**Verdict: CONFIRMED.** Once low-volume noise is filtered out, CTR drops cleanly as position
gets worse (0.23 → 0.20 → 0.17 → 0.08 → 0.00 — `top_3` sits just under `page_1`, which is close
enough to still call the direction real). This is the real relationship I need before I can
claim a page's CTR is "underperforming" — I compare each page's own CTR against the
floor-filtered median for its own position tier.

### The rule, in plain words

A page is worth prioritizing for refresh if it has real visibility (`impressions_90d >= 300` —
the volume floor Signal 2 showed matters), it sits in the specific staleness window Signal 1
showed elevated risk in (`freshness_tier == "91-180"`), and — separately — whether its actual
CTR falls short of what pages at its own position tier normally get (from Signal 2's
floor-filtered expected-CTR table). That CTR gap is what splits the flagged pages into two
reason codes.

### Reason codes (one per row)

- `stale_visible_ctr_gap` — stale + visible + CTR below its tier's expected rate → action
  `refresh_and_fix_ctr`
- `stale_but_visible` — stale + visible but CTR at/above expected → action `refresh_review`
- `not_flagged` — everything else → action `no_action`

### Score (transparent, no fitted weights)

```
score = stale * visible * impressions_90d * (1 + ctr_underperforming)
```
Readable on purpose: staleness and visibility gate whether a page is considered at all: a CTR
gap doubles its weight once it's in the running, but visibility (raw reach) is what orders the
queue inside that group.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# --- Signal check 1: staleness vs decline rate ---
sig1 = df.groupby("freshness_tier").agg(
    n=("is_declining_label", "size"),
    decline_rate=("is_declining_label", "mean"),
)
sig1["decline_rate_pct"] = (sig1["decline_rate"] * 100).round(1)
print("Signal 1 — freshness_tier vs decline rate (base rate = {:.1f}%)".format(
    df["is_declining_label"].mean() * 100))
print(sig1[["n", "decline_rate_pct"]])
print()

# --- Signal check 2: CTR vs position, with a volume floor ---
raw = df.groupby("position_tier").agg(n=("ctr", "size"), median_ctr=("ctr", "median")).round(3)
print("Signal 2 (no volume floor — misleading) — position_tier vs median CTR")
print(raw)
print()

floor = df[df["impressions_90d"] >= 300]
expected_ctr_by_tier = floor.groupby("position_tier")["ctr"].median()
floored = floor.groupby("position_tier").agg(n=("ctr", "size"), median_ctr=("ctr", "median")).round(3)
print("Signal 2 (impressions_90d >= 300 floor) — position_tier vs median CTR")
print(floored)
print()
print("Expected CTR by tier (used as the rule's per-tier baseline):")
print(expected_ctr_by_tier)

# --- Rule inputs (no label-derived, no future-window columns) ---
df["expected_ctr"] = df["position_tier"].map(expected_ctr_by_tier)
stale = df["freshness_tier"] == "91-180"
visible = df["impressions_90d"] >= 300
ctr_underperforming = df["ctr"] < df["expected_ctr"]

df["score"] = (
    stale.astype(int) * visible.astype(int) * df["impressions_90d"] * (1 + ctr_underperforming.astype(int))
).round(1)

df["reason_code"] = "not_flagged"
df.loc[stale & visible & ctr_underperforming, "reason_code"] = "stale_visible_ctr_gap"
df.loc[stale & visible & ~ctr_underperforming, "reason_code"] = "stale_but_visible"

df["action_label"] = "no_action"
df.loc[df["reason_code"] == "stale_visible_ctr_gap", "action_label"] = "refresh_and_fix_ctr"
df.loc[df["reason_code"] == "stale_but_visible", "action_label"] = "refresh_review"

print()
print("Flagged rows (score > 0):", int((df["score"] > 0).sum()), "/", len(df))
print(df["reason_code"].value_counts())
print(df["action_label"].value_counts())

Signal 1 — freshness_tier vs decline rate (base rate = 54.2%)
                    n  decline_rate_pct
freshness_tier                         
0-30            20480              51.1
181+              174              47.1
31-90             175              58.9
91-180           9171              61.1

Signal 2 (no volume floor — misleading) — position_tier vs median CTR
                   n  median_ctr
position_tier                   
deep            1319        0.00
page_1         11814        0.16
page_3_5        7242        0.03
striking        7304        0.11
top_3           2321        0.00

Signal 2 (impressions_90d >= 300 floor) — position_tier vs median CTR
                  n  median_ctr
position_tier                  
deep            562        0.00
page_1         7623        0.23
page_3_5       5004        0.08
striking       5078        0.17
top_3           485        0.20

Expected CTR by tier (used as the rule's per-tier baseline):
position_tier
deep        0.00
page_1  

## 2. Build the ranked queue (writes the CSV)

Rank every row by `score` (descending), keep the columns a reviewer actually needs, and write
the full ranked queue to `work/outputs/baseline_action_score.csv`. As an honest sanity check —
not a formal requirement of this card, but cheap and worth knowing — I also print precision@10
and precision@20 against `is_declining_label`, next to the base rate, so a bare "80% precision"
can't be read as impressive without knowing 54.2% is already the coin-flip floor.

In [2]:
import numpy as np
import os

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

out_cols = [
    "rank", "content_id", "client_id", "score", "reason_code", "action_label",
    "impressions_90d", "ctr", "expected_ctr", "position_tier", "freshness_tier",
    "days_since_last_update",
]
os.makedirs("work/outputs", exist_ok=True)
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote", len(ranked), "rows to work/outputs/baseline_action_score.csv")
print(ranked[out_cols].head(3))

def precision_at_k(labels, k):
    return float(np.asarray(labels)[:k].mean())

base_rate = df["is_declining_label"].mean()
p10 = precision_at_k(ranked["is_declining_label"].values, 10)
p20 = precision_at_k(ranked["is_declining_label"].values, 20)
print()
print(f"Base rate (declining, all 30,000 rows): {base_rate*100:.1f}%")
print(f"precision@10 of the ranked queue: {p10*100:.1f}%")
print(f"precision@20 of the ranked queue: {p20*100:.1f}%")

Wrote 30000 rows to work/outputs/baseline_action_score.csv
   rank            content_id          client_id    score  \
0     1  content_5fe46e04994d  client_4e07408562  1035430   
1     2  content_cb112fce36be  client_19581e27de   619820   
2     3  content_36ff89c8214e  client_19581e27de   590194   

             reason_code         action_label  impressions_90d   ctr  \
0  stale_visible_ctr_gap  refresh_and_fix_ctr           517715  0.14   
1  stale_visible_ctr_gap  refresh_and_fix_ctr           309910  0.16   
2  stale_visible_ctr_gap  refresh_and_fix_ctr           295097  0.05   

   expected_ctr position_tier freshness_tier  days_since_last_update  
0          0.23        page_1         91-180                     104  
1          0.23        page_1         91-180                     104  
2          0.23        page_1         91-180                     104  

Base rate (declining, all 30,000 rows): 54.2%
precision@10 of the ranked queue: 50.0%
precision@20 of the ranked queue: 65

## 3. Top-10 review

One line each: the action, why it landed here, and what would make it wrong.

1. **content_5fe46e04994d** (`refresh_and_fix_ctr`) — 517,715 impressions on `page_1`, CTR
   0.14% vs an expected 0.23%, untouched for 104 days: the single biggest visible CTR gap in
   the set. *Wrong if* the low CTR is a search-intent mismatch (title promises something the
   page doesn't deliver) rather than staleness — a refresh wouldn't fix that.
2. **content_cb112fce36be** (`refresh_and_fix_ctr`) — 309,910 impressions, `page_1`, CTR 0.16%
   vs 0.23% expected, same 104-day staleness window. *Wrong if* this page has a title/meta
   experiment mid-run — refreshing now interrupts real data collection instead of helping it.
3. **content_36ff89c8214e** (`refresh_and_fix_ctr`) — largest CTR gap by ratio: 0.05% vs 0.23%
   expected, more than 4x under, 295,097 impressions. *Wrong if* a SERP feature (snippet,
   People Also Ask) is stealing the click above this result — rewriting copy won't fix that.
4. **content_b28d1efd668f** (`refresh_and_fix_ctr`) — `page_3_5`, CTR 0.06% vs 0.08% expected:
   a small gap, ranked mostly on staleness + volume. *Wrong if* this page and #5 below (same
   client, same tier, near-identical numbers) are really one content cluster — fixing one might
   fix both, so flagging both separately double-counts the actual work.
5. **content_813e88069237** (`refresh_and_fix_ctr`) — same client and pattern as #4. *Wrong for
   the same reason*: likely duplicate/near-duplicate work with #4, not two independent picks.
6. **content_2dba2b1f9536** (`refresh_review`) — CTR 0.21% is already *above* the 0.08%
   expected for `page_3_5`, so this isn't a CTR problem, just stale + visible. *Wrong if* this
   page's traffic is naturally seasonal and about to decline anyway regardless of a refresh.
7. **content_c8e9d6ab9013** (`refresh_and_fix_ctr`) — CTR is exactly 0.00% on `page_1` with
   208,678 impressions — the worst possible outcome in the set. *Wrong if* this is a tracking
   or data-pipeline gap rather than a real content problem — 0.00% at that volume is unusual
   enough to warrant a manual check before assuming it's fixable by editing the page.
8. **content_a7427266c305** (`refresh_and_fix_ctr`) — CTR 0.11% vs 0.23% expected, `page_1`,
   201,111 impressions. *Wrong if* this page ranks for a branded/navigational query, where users
   don't click through by nature — the CTR-fix logic assumes informational intent.
9. **content_33b4dceecad1** (`refresh_and_fix_ctr`) — CTR 0.16% vs 0.23% expected: the smallest
   gap of the flagged group, ranked here mostly on raw volume (181,574 impressions). *Wrong if*
   this gap is within normal noise for the sample — it's the weakest CTR case in the top 10.
10. **content_2c2606c5d176** (`refresh_review`) — CTR 0.53%, more than double the 0.23%
    expected for `page_1` — a strong performer, flagged only for staleness. *Wrong to treat this
    as a CTR fix at all*; the right action is a light refresh to protect what's working, not a
    rewrite.

In [3]:
top10 = ranked[out_cols].head(10)
top10

,rank,content_id,client_id,score,reason_code,action_label,impressions_90d,ctr,expected_ctr,position_tier,freshness_tier,days_since_last_update
0,1,content_5fe46e04994d,client_4e07408562,1035430,stale_visible_ctr_gap,refresh_and_fix_ctr,517715,0.14,0.23,page_1,91-180,104
1,2,content_cb112fce36be,client_19581e27de,619820,stale_visible_ctr_gap,refresh_and_fix_ctr,309910,0.16,0.23,page_1,91-180,104
2,3,content_36ff89c8214e,client_19581e27de,590194,stale_visible_ctr_gap,refresh_and_fix_ctr,295097,0.05,0.23,page_1,91-180,104
3,4,content_b28d1efd668f,client_6208ef0f77,573216,stale_visible_ctr_gap,refresh_and_fix_ctr,286608,0.06,0.08,page_3_5,91-180,104
4,5,content_813e88069237,client_6208ef0f77,467122,stale_visible_ctr_gap,refresh_and_fix_ctr,233561,0.06,0.08,page_3_5,91-180,104
5,6,content_2dba2b1f9536,client_6208ef0f77,443434,stale_but_visible,refresh_review,443434,0.21,0.08,page_3_5,91-180,104
6,7,content_c8e9d6ab9013,client_19581e27de,417356,stale_visible_ctr_gap,refresh_and_fix_ctr,208678,0.00,0.23,page_1,91-180,104
7,8,content_a7427266c305,client_19581e27de,402222,stale_visible_ctr_gap,refresh_and_fix_ctr,201111,0.11,0.23,page_1,91-180,104
8,9,content_33b4dceecad1,client_19581e27de,363148,stale_visible_ctr_gap,refresh_and_fix_ctr,181574,0.16,0.23,page_1,91-180,104
9,10,content_2c2606c5d176,client_19581e27de,347399,stale_but_visible,refresh_review,347399,0.53,0.23,page_1,91-180,104


## 4. Weak picks + leakage check

### Weak picks

- **Client concentration.** 6 of the top 10 (and 17 of the top 20) come from just 2 clients
  (`client_19581e27de`, `client_6208ef0f77`). Because `score` is `impressions_90d`-driven, the
  rule structurally favors clients with the most raw traffic, not clients with the most urgent
  problem relative to their own size. A client with 10x less traffic overall could have a page
  that's collapsing and it would never surface near the top of this queue. A real version of
  this rule should probably rank within-client, or normalize the score by each client's own
  traffic scale, before treating it as a single global priority list.
- **Same-day staleness cluster.** All 10 rows share `days_since_last_update == 104` exactly.
  That's either a real batch content event (a CMS migration or bulk-publish date) or an
  artifact of how this teaching slice was generated — either way, 10 "independent" top picks
  that all share one exact date is a pattern worth checking before trusting them as 10 separate
  judgments.
- **Near-duplicate picks.** #4 and #5 (both `client_6208ef0f77`, same tier, near-identical
  score/CTR/impressions) look like they could be the same content cluster counted twice, which
  would mean the top 10 really contains closer to 9 distinct actions.

### Leakage check

The rule's inputs are `freshness_tier` (from `days_since_last_update`), `impressions_90d`,
`ctr`, and `position_tier`. None of these touch the label path:

- `trend_direction` and `trend_pct` (the label source for `is_declining_label`) are **not**
  used anywhere in `score`, `reason_code`, or `action_label` — confirmed by inspecting the
  three lines that build them, printed below.
- The raw `avg_position` column was **not** used directly (its `0 = no data` gotcha doesn't
  apply here) — I used the pre-built `position_tier` category instead.
- No `*_last_30d` / `*_prev_30d` trend-window columns were touched — the rule only reads
  90-day totals and static content properties, so there's no future-window input to leak.

In [4]:
import inspect

rule_cols_used = {"freshness_tier", "impressions_90d", "ctr", "position_tier", "expected_ctr"}
label_cols = {"trend_direction", "trend_pct", "is_declining_label"}
print("Rule input columns:", sorted(rule_cols_used))
print("Overlap with label-derived columns:", rule_cols_used & label_cols, "(should be empty)")

print()
print("Client concentration — top 10:")
print(top10["client_id"].value_counts())
print()
print("Client concentration — top 20:")
print(ranked[out_cols].head(20)["client_id"].value_counts())
print()
print("days_since_last_update values in top 10:", sorted(top10["days_since_last_update"].unique()))

Rule input columns: ['ctr', 'expected_ctr', 'freshness_tier', 'impressions_90d', 'position_tier']
Overlap with label-derived columns: set() (should be empty)

Client concentration — top 10:
client_id
client_19581e27de    6
client_6208ef0f77    3
client_4e07408562    1
Name: count, dtype: int64

Client concentration — top 20:
client_id
client_19581e27de    11
client_6208ef0f77     6
client_4e07408562     2
client_3fdba35f04     1
Name: count, dtype: int64

days_since_last_update values in top 10: [np.int64(104)]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.